# 01 - Qdrant desde cero

Objetivo: entender `Qdrant` como componente aislado, sin LangChain y sin pipeline RAG.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv(Path("..").resolve() / ".env")

QDRANT_PORT = os.getenv("QDRANT_PORT", "6333")
QDRANT_URL = f"http://127.0.0.1:{QDRANT_PORT}"
COLLECTION_NAME = "sandbox_manual_vectors"

client = QdrantClient(url=QDRANT_URL)
print({"QDRANT_URL": QDRANT_URL, "COLLECTION_NAME": COLLECTION_NAME})


In [ ]:
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
)

points = [
    PointStruct(id=1, vector=[0.9, 0.1, 0.0, 0.0], payload={"label": "devoluciones"}),
    PointStruct(id=2, vector=[0.0, 0.9, 0.1, 0.0], payload={"label": "montaje"}),
    PointStruct(id=3, vector=[0.0, 0.1, 0.9, 0.2], payload={"label": "envios"}),
]

client.upsert(collection_name=COLLECTION_NAME, points=points)
client.get_collection(COLLECTION_NAME)


In [ ]:
query_vector = [0.85, 0.15, 0.0, 0.0]
hits = client.search(collection_name=COLLECTION_NAME, query_vector=query_vector, limit=2)

for hit in hits:
    print({
        "id": hit.id,
        "score": round(hit.score, 4),
        "payload": hit.payload,
    })
